## This code is for the data generation of a) Ellipses on a square domain (train/test and defect data), b) ellipses generated on a circular subdomain (train/test and defect data) 

### (a) Square domain data:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import os
import time
from numba import cuda, jit
import math

# Start timing
start_time = time.time()

# Constants
DOMAIN_SIZE = 1.6  # Square domain
P_VALUES = np.round(np.array([0, 0.1, 0.4, 0.6, 1.0]), decimals=1) # Use these p-values to not over-represent random configs or nematic configs
IMAGES_PER_P = 100 # match with num_images below

# CUDA configuration - optimized for modern GPUs
THREADS_PER_BLOCK = 256

def setup_cuda():
    """Initialize CUDA for optimal performance"""
    cuda.select_device(0)

@cuda.jit(device=True)
def device_is_inside_ellipse_optimized(x, y, ellipse_params, min_distance):
    """Optimized ellipse collision detection with precomputed values"""
    x_center, y_center, a, b, angle = ellipse_params
    cos_angle = math.cos(angle)
    sin_angle = math.sin(angle)
    x_rot = (x - x_center) * cos_angle + (y - y_center) * sin_angle
    y_rot = -(x - x_center) * sin_angle + (y - y_center) * cos_angle
    
    # Use multiplication instead of division for better performance
    if min_distance > 0:
        effective_a = a + min_distance
        effective_b = b + min_distance
        # Compare squared values to avoid sqrt
        return (x_rot * x_rot) / (effective_a * effective_a) + (y_rot * y_rot) / (effective_b * effective_b) <= 1.0
    else:
        return (x_rot * x_rot) / (a * a) + (y_rot * y_rot) / (b * b) <= 1.0

@cuda.jit
def check_overlaps_kernel_optimized(ellipses, results, new_ellipse_idx, min_distance):
    """Optimized kernel with shared memory and precomputation"""
    i = cuda.grid(1)
    total_ellipses = ellipses.shape[0]
    
    if i < total_ellipses - 1:
        # Shared memory for the new ellipse - reduces global memory access
        new_ellipse_shared = cuda.shared.array(5, dtype=np.float32)
        tid = cuda.threadIdx.x
        
        # Load new ellipse into shared memory (first 5 threads do this)
        if tid < 5:
            new_ellipse_shared[tid] = ellipses[new_ellipse_idx, tid]
        cuda.syncthreads()
        
        # Get existing ellipse parameters
        existing_ellipse = (
            ellipses[i, 0], ellipses[i, 1], ellipses[i, 2],
            ellipses[i, 3], ellipses[i, 4]
        )

        overlap_found = False
        num_points = 100  # Reduced from original for speed, adjust as needed

        # Precompute rotation values once for the new ellipse
        new_cos_a = math.cos(new_ellipse_shared[4])
        new_sin_a = math.sin(new_ellipse_shared[4])
        
        # Precompute rotation values once for the existing ellipse
        existing_cos_a = math.cos(existing_ellipse[4])
        existing_sin_a = math.sin(existing_ellipse[4])

        # Check points on new ellipse against existing ellipses
        for j in range(num_points):
            theta = 2 * math.pi * j / num_points
            cos_t = math.cos(theta)
            sin_t = math.sin(theta)

            # Transform point on new ellipse
            x = (new_ellipse_shared[0] + 
                 new_ellipse_shared[2] * cos_t * new_cos_a - 
                 new_ellipse_shared[3] * sin_t * new_sin_a)
            y = (new_ellipse_shared[1] + 
                 new_ellipse_shared[2] * cos_t * new_sin_a + 
                 new_ellipse_shared[3] * sin_t * new_cos_a)

            if device_is_inside_ellipse_optimized(x, y, existing_ellipse, min_distance):
                overlap_found = True
                break

        # Check points on existing ellipse against new ellipse
        if not overlap_found:
            for j in range(num_points):
                theta = 2 * math.pi * j / num_points
                cos_t = math.cos(theta)
                sin_t = math.sin(theta)

                # Transform point on existing ellipse
                x = (existing_ellipse[0] + 
                     existing_ellipse[2] * cos_t * existing_cos_a - 
                     existing_ellipse[3] * sin_t * existing_sin_a)
                y = (existing_ellipse[1] + 
                     existing_ellipse[2] * cos_t * existing_sin_a + 
                     existing_ellipse[3] * sin_t * existing_cos_a)

                if device_is_inside_ellipse_optimized(x, y, 
                    (new_ellipse_shared[0], new_ellipse_shared[1], 
                     new_ellipse_shared[2], new_ellipse_shared[3], 
                     new_ellipse_shared[4]), min_distance):
                    overlap_found = True
                    break

        if overlap_found:
            results[i] = 1

@cuda.jit
def initialize_array_kernel(array, value):
    """Kernel to initialize device array with a value"""
    i = cuda.grid(1)
    if i < array.size:
        array[i] = value

def check_overlaps_cuda_optimized(ellipses, new_ellipse_idx, min_distance=0.0):
    """Optimized overlap checking with better memory management"""
    # Ensure contiguous array for better memory access
    ellipses_array = np.ascontiguousarray(ellipses, dtype=np.float32)
    d_ellipses = cuda.to_device(ellipses_array)
    d_results = cuda.device_array(len(ellipses)-1, dtype=np.int32)
    
    # Initialize results to 0 using a kernel
    blocks_per_grid_init = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    initialize_array_kernel[blocks_per_grid_init, THREADS_PER_BLOCK](d_results, 0)
    
    blocks_per_grid = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    
    # Use the optimized kernel
    check_overlaps_kernel_optimized[blocks_per_grid, THREADS_PER_BLOCK](
        d_ellipses, d_results, new_ellipse_idx, min_distance
    )
    
    results = d_results.copy_to_host()
    return np.any(results == 1)

@jit(nopython=True)
def calculate_q_tensor_optimized(angles):
    """Optimized Q-tensor calculation"""
    Q11 = 0.0
    Q12 = 0.0
    n = len(angles)

    for angle in angles:
        cos_theta = math.cos(2 * angle)  # Use double angle for direct calculation
        sin_theta = math.sin(2 * angle)
        Q11 += cos_theta
        Q12 += sin_theta

    Q11 /= (2.0 * n)  # Adjust for the double angle formula
    Q12 /= (2.0 * n)
    return Q11, Q12, Q12, -Q11

@jit(nopython=True)
def check_ellipse_in_bounds(x_center, y_center, major_axis, minor_axis, angle, half_size):
    """Check if ellipse is completely within square domain bounds"""
    # Check points around the ellipse perimeter (can increase, but this seems to be okay)
    num_check_points = 36
    for i in range(num_check_points):
        theta = 2 * math.pi * i / num_check_points
        cos_t = math.cos(theta)
        sin_t = math.sin(theta)
        cos_a = math.cos(angle)
        sin_a = math.sin(angle)
        
        # Calculate point on ellipse boundary
        x = x_center + major_axis * cos_t * cos_a - minor_axis * sin_t * sin_a
        y = y_center + major_axis * cos_t * sin_a + minor_axis * sin_t * cos_a
        
        # Check if point is outside square domain
        if abs(x) > half_size or abs(y) > half_size:
            return False
    return True

def ellipse_params_optimized(major_axis, minor_axis, num_ellipses, p, domain_size=DOMAIN_SIZE, seed_offset=0, min_distance=0.01):
    """Optimized ellipse generation on square domain"""
    ellipses = []
    angles = []
    max_attempts = 500

    full_range_upper = math.pi - 0.01
    biased_upper = (1 - p) * full_range_upper + p * (math.pi / 5)
    base_rotation = (seed_offset * 137.5) % math.pi
    
    # Square domain boundaries
    half_size = domain_size / 2.0

    for attempt in range(max_attempts):
        if len(ellipses) >= num_ellipses:
            break

        # Generate uniform random position in square domain
        x_center = np.random.uniform(-half_size, half_size)
        y_center = np.random.uniform(-half_size, half_size)
        
        angle = (np.random.uniform(0, biased_upper) + base_rotation) % math.pi
        
        # Check if ellipse is completely within bounds
        if not check_ellipse_in_bounds(x_center, y_center, major_axis, minor_axis, angle, half_size):
            continue
        
        new_ellipse = (x_center, y_center, major_axis, minor_axis, angle)

        if len(ellipses) > 0:
            # Create temporary list for checking
            temp_ellipses = ellipses + [new_ellipse]
            temp_array = np.array(temp_ellipses, dtype=np.float32)
            
            # Use optimized overlap checking
            overlaps = check_overlaps_cuda_optimized(temp_array, len(temp_ellipses)-1, min_distance)
        else:
            overlaps = False

        if not overlaps:
            ellipses.append(new_ellipse)
            angles.append(angle)

    return ellipses, angles

def convert_to_black_white_optimized(image_array, threshold=128):
    """
    Optimized black and white conversion
    """
    if len(image_array.shape) == 3:
        # Faster grayscale conversion using integer arithmetic
        image_gray = np.dot(image_array[...,:3], [299, 587, 114]) // 1000
    else:
        image_gray = image_array
    
    # Normalize efficiently
    if image_gray.max() > 1.0:
        min_val = image_gray.min()
        max_val = image_gray.max()
        if max_val > min_val:
            image_gray = ((image_gray - min_val) * 255 / (max_val - min_val)).astype(np.uint8)
    
    # Binary threshold
    binary_image = np.where(image_gray >= threshold, 255, 0).astype(np.uint8)
    
    return binary_image

def plot_ellipses_optimized(ellipses, seed, Q_values, execution_time, output_subfolder, N=250): ##------------------------ SIZE OF IMAGE HERE
    """Optimized ellipse plotting on square domain"""
    dpi = 100
    figsize_inches = N / dpi
    fig, ax = plt.subplots(figsize=(figsize_inches, figsize_inches), dpi=dpi)

    # Use collection for faster rendering
    from matplotlib.collections import PatchCollection
    patches = []
    
    for x, y, a, b, angle in ellipses:
        ell = plt.matplotlib.patches.Ellipse((x, y), 2*a, 2*b,
                                             angle=np.degrees(angle),
                                             fill=True, edgecolor='white',  # Changed from 'black' to 'white'
                                             facecolor='white')  # Changed from 'black' to 'white'
        patches.append(ell)

    collection = PatchCollection(patches, match_original=True)
    ax.add_collection(collection)

    # Square domain limits
    half_size = DOMAIN_SIZE / 2.0
    ax.set_xlim(-half_size, half_size)
    ax.set_ylim(-half_size, half_size)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    ax.set_facecolor('black')  # Changed from 'white' to 'black' for background

    filename = f'p{p:.1f}_seed_{seed}_Q11_{Q_values[0]:.4f}_Q12_{Q_values[1]:.4f}.png'
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # Save the initial image
    temp_path = os.path.join(output_subfolder, 'temp_' + filename)
    plt.savefig(temp_path, dpi=dpi, bbox_inches='tight', pad_inches=0, facecolor='black')  # Changed from 'white' to 'black'
    plt.close(fig)
    
    # Convert to pure black and white
    img = Image.open(temp_path)
    img_array = np.array(img)
    
    # Use optimized conversion
    bw_array = convert_to_black_white_optimized(img_array)
    
    # Save final black and white image
    bw_img = Image.fromarray(bw_array)
    final_path = os.path.join(output_subfolder, filename)
    bw_img.save(final_path, optimize=True, quality=85)
    
    # Remove temporary file
    os.remove(temp_path)

def generate_images_optimized(num_images, major_axis, minor_axis, num_ellipses, p, p_idx, min_distance=0.01):
    """Optimized image generation with better resource management"""
    output_dir = 'output_ellipses'
    output_subfolder = os.path.join(output_dir, 'MoreSquareTrain/TestData') ##### name change 
    os.makedirs(output_subfolder, exist_ok=True)

    Q_tensor_list = []

    for image_idx in range(1, num_images + 1):
        # Use this seed naming convention
        p_index = int(p * 10)
        seed = 0 + p_index * IMAGES_PER_P + image_idx  ########  ------------------------- for test data offset replace 0 with a larger number
        
        np.random.seed(seed)



        
        # Use optimized ellipse generation
        ellipses, angles = ellipse_params_optimized(
            major_axis, minor_axis, num_ellipses, p, 
            seed_offset=seed, min_distance=min_distance
        )
        
        # Use optimized Q-tensor calculation
        Q11, Q12, Q21, Q22 = calculate_q_tensor_optimized(angles)
        Q_tensor_list.append((seed, Q11, Q12, Q21, Q22))

        execution_time = time.time() - start_time
        
        # Use optimized plotting
        plot_ellipses_optimized(ellipses, seed, (Q11, Q12), 
                               execution_time, output_subfolder, N=250) ########## --------------------------------------SIZE OF IMAGE HERE
        
        # Progress reporting
        if image_idx % 10 == 0:
            print(f"  Generated {image_idx}/{num_images} images for p={p:.1f}")

    return Q_tensor_list

# Parameters
PARAMS = {
    'major_axis': 0.11,
    'minor_axis': 0.01,
    'num_ellipses': 100,
    'num_images': 100,  # num_images*(how many p-values) = total images, i.e., gives num_images number of data per p-value
    'min_distance': 0.01  # Minimum distance between ellipses (denser configurations if small)
}

# Main execution
if __name__ == '__main__':
    # Initialize CUDA for optimal performance
    setup_cuda()
    
    print(f"Starting optimized ellipse generation on square domain")
    print(f"Domain size: {DOMAIN_SIZE} x {DOMAIN_SIZE}")
    print(f"Minimum distance: {PARAMS['min_distance']}")
    print(f"Using CUDA with {THREADS_PER_BLOCK} threads per block")
    
    total_start_time = time.time()
    
    for p_idx, p in enumerate(P_VALUES):
        p_start_time = time.time()
        print(f"Generating images for p = {p:.1f}...")
        
        Q_tensors = generate_images_optimized(**PARAMS, p=p, p_idx=p_idx)
        
        p_time = time.time() - p_start_time
        print(f"Completed p = {p:.1f} in {p_time:.2f} seconds")
        print(f"  Generated {len(Q_tensors)} configurations")
    
    total_time = time.time() - total_start_time
    print(f"\nTotal execution time: {total_time:.2f} seconds")
    print(f"Average time per p-value: {total_time/len(P_VALUES):.2f} seconds")

### (a) Defect-type data on a square domain:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import os
import time
from numba import cuda, jit
import math

# Start timing
start_time = time.time()

# Constants
DOMAIN_SIZE = 1.6 
P_VALUES = np.round(np.array([0, 0.1, 0.4, 0.6, 1.0]), decimals=1)
IMAGES_PER_P = 100 

# CUDA configuration - optimized for modern GPUs
THREADS_PER_BLOCK = 256

def setup_cuda():
    """Initialize CUDA for optimal performance"""
    try:
        cuda.select_device(0)
    except:
        print("CUDA device not found or already active")

@cuda.jit(device=True)
def device_is_inside_ellipse_optimized(x, y, ellipse_params, min_distance):
    """Optimized ellipse collision detection with precomputed values"""
    x_center, y_center, a, b, angle = ellipse_params
    cos_angle = math.cos(angle)
    sin_angle = math.sin(angle)
    x_rot = (x - x_center) * cos_angle + (y - y_center) * sin_angle
    y_rot = -(x - x_center) * sin_angle + (y - y_center) * cos_angle
    
    # Use multiplication instead of division for better performance
    if min_distance > 0:
        effective_a = a + min_distance
        effective_b = b + min_distance
        # Compare squared values to avoid sqrt
        return (x_rot * x_rot) / (effective_a * effective_a) + (y_rot * y_rot) / (effective_b * effective_b) <= 1.0
    else:
        return (x_rot * x_rot) / (a * a) + (y_rot * y_rot) / (b * b) <= 1.0

@cuda.jit
def check_overlaps_kernel_optimized(ellipses, results, new_ellipse_idx, min_distance):
    """Optimized kernel with shared memory and precomputation"""
    i = cuda.grid(1)
    total_ellipses = ellipses.shape[0]
    
    if i < total_ellipses - 1:
        # Shared memory for the new ellipse - reduces global memory access
        new_ellipse_shared = cuda.shared.array(5, dtype=np.float32)
        tid = cuda.threadIdx.x
        
        # Load new ellipse into shared memory (first 5 threads do this)
        if tid < 5:
            new_ellipse_shared[tid] = ellipses[new_ellipse_idx, tid]
        cuda.syncthreads()
        
        # Get existing ellipse parameters
        existing_ellipse = (
            ellipses[i, 0], ellipses[i, 1], ellipses[i, 2],
            ellipses[i, 3], ellipses[i, 4]
        )

        overlap_found = False
        num_points = 75  # Moderate point count for performance

        # Precompute rotation values once for the new ellipse
        new_cos_a = math.cos(new_ellipse_shared[4])
        new_sin_a = math.sin(new_ellipse_shared[4])
        
        # Precompute rotation values once for the existing ellipse
        existing_cos_a = math.cos(existing_ellipse[4])
        existing_sin_a = math.sin(existing_ellipse[4])

        # Check points on new ellipse against existing ellipses
        for j in range(num_points):
            theta = 2 * math.pi * j / num_points
            cos_t = math.cos(theta)
            sin_t = math.sin(theta)

            # Transform point on new ellipse
            x = (new_ellipse_shared[0] + 
                 new_ellipse_shared[2] * cos_t * new_cos_a - 
                 new_ellipse_shared[3] * sin_t * new_sin_a)
            y = (new_ellipse_shared[1] + 
                 new_ellipse_shared[2] * cos_t * new_sin_a + 
                 new_ellipse_shared[3] * sin_t * new_cos_a)

            if device_is_inside_ellipse_optimized(x, y, existing_ellipse, min_distance):
                overlap_found = True
                break

        # Check points on existing ellipse against new ellipse
        if not overlap_found:
            for j in range(num_points):
                theta = 2 * math.pi * j / num_points
                cos_t = math.cos(theta)
                sin_t = math.sin(theta)

                # Transform point on existing ellipse
                x = (existing_ellipse[0] + 
                     existing_ellipse[2] * cos_t * existing_cos_a - 
                     existing_ellipse[3] * sin_t * existing_sin_a)
                y = (existing_ellipse[1] + 
                     existing_ellipse[2] * cos_t * existing_sin_a + 
                     existing_ellipse[3] * sin_t * existing_cos_a)

                if device_is_inside_ellipse_optimized(x, y, 
                    (new_ellipse_shared[0], new_ellipse_shared[1], 
                     new_ellipse_shared[2], new_ellipse_shared[3], 
                     new_ellipse_shared[4]), min_distance):
                    overlap_found = True
                    break

        if overlap_found:
            results[i] = 1

@cuda.jit
def initialize_array_kernel(array, value):
    """Kernel to initialize device array with a value"""
    i = cuda.grid(1)
    if i < array.size:
        array[i] = value

def check_overlaps_cuda_optimized(ellipses, new_ellipse_idx, min_distance=0.0):
    """Optimized overlap checking with better memory management"""
    # Ensure contiguous array for better memory access
    ellipses_array = np.ascontiguousarray(ellipses, dtype=np.float32)
    d_ellipses = cuda.to_device(ellipses_array)
    d_results = cuda.device_array(len(ellipses)-1, dtype=np.int32)
    
    # Initialize results to 0 using a kernel
    blocks_per_grid_init = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    initialize_array_kernel[blocks_per_grid_init, THREADS_PER_BLOCK](d_results, 0)
    
    blocks_per_grid = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    
    # Use the optimized kernel
    check_overlaps_kernel_optimized[blocks_per_grid, THREADS_PER_BLOCK](
        d_ellipses, d_results, new_ellipse_idx, min_distance
    )
    
    results = d_results.copy_to_host()
    return np.any(results == 1)

@jit(nopython=True)
def calculate_q_tensor_optimized(angles):
    """Optimized Q-tensor calculation"""
    Q11 = 0.0
    Q12 = 0.0
    n = len(angles)

    for angle in angles:
        cos_theta = math.cos(2 * angle)  # Use double angle for direct calculation
        sin_theta = math.sin(2 * angle)
        Q11 += cos_theta
        Q12 += sin_theta

    Q11 /= (2.0 * n)  # Adjust for the double angle formula
    Q12 /= (2.0 * n)
    return Q11, Q12, Q12, -Q11

@jit(nopython=True)
def check_ellipse_in_bounds(x_center, y_center, major_axis, minor_axis, angle, half_size):
    """Check if ellipse is completely within square domain bounds"""
    # Check points around the ellipse perimeter
    num_check_points = 36
    for i in range(num_check_points):
        theta = 2 * math.pi * i / num_check_points
        cos_t = math.cos(theta)
        sin_t = math.sin(theta)
        cos_a = math.cos(angle)
        sin_a = math.sin(angle)
        
        # Calculate point on ellipse boundary
        x = x_center + major_axis * cos_t * cos_a - minor_axis * sin_t * sin_a
        y = y_center + major_axis * cos_t * sin_a + minor_axis * sin_t * cos_a
        
        # Check if point is outside square domain
        if abs(x) > half_size or abs(y) > half_size:
            return False
    return True

def ellipse_params_optimized(major_axis, minor_axis, num_ellipses, p, domain_size=DOMAIN_SIZE, seed_offset=0, min_distance=0.01):
    """
    Optimized ellipse generation on square domain with HEDGEHOG DEFECT orientation.
    """
    ellipses = []
    angles = []
    max_attempts = 700 # Increased attempts for defects

    # Square domain boundaries
    half_size = domain_size / 2.0

    for attempt in range(max_attempts):
        if len(ellipses) >= num_ellipses:
            break

        # Generate uniform random position in square domain
        x_center = np.random.uniform(-half_size, half_size)
        y_center = np.random.uniform(-half_size, half_size)
        
        # --- DEFECT LOGIC MODIFICATION ---
        # Instead of random orientation, align radially
        angle = math.atan2(y_center, x_center)
        # ---------------------------------
        
        # Check if ellipse is completely within bounds
        if not check_ellipse_in_bounds(x_center, y_center, major_axis, minor_axis, angle, half_size):
            continue
        
        new_ellipse = (x_center, y_center, major_axis, minor_axis, angle)

        if len(ellipses) > 0:
            # Create temporary list for checking
            temp_ellipses = ellipses + [new_ellipse]
            temp_array = np.array(temp_ellipses, dtype=np.float32)
            
            # Use optimized overlap checking
            overlaps = check_overlaps_cuda_optimized(temp_array, len(temp_ellipses)-1, min_distance)
        else:
            overlaps = False

        if not overlaps:
            ellipses.append(new_ellipse)
            angles.append(angle)

    return ellipses, angles

def convert_to_black_white_optimized(image_array, threshold=128):
    """
    Optimized black and white conversion
    """
    if len(image_array.shape) == 3:
        # Faster grayscale conversion using integer arithmetic
        image_gray = np.dot(image_array[...,:3], [299, 587, 114]) // 1000
    else:
        image_gray = image_array
    
    # Normalize efficiently
    if image_gray.max() > 1.0:
        min_val = image_gray.min()
        max_val = image_gray.max()
        if max_val > min_val:
            image_gray = ((image_gray - min_val) * 255 / (max_val - min_val)).astype(np.uint8)
    
    # Binary threshold
    binary_image = np.where(image_gray >= threshold, 255, 0).astype(np.uint8)
    
    return binary_image

def plot_ellipses_optimized(ellipses, seed, Q_values, execution_time, output_subfolder, N=250):
    """Optimized ellipse plotting on square domain with Defect Style (White on Black)"""
    dpi = 100
    figsize_inches = N / dpi
    fig, ax = plt.subplots(figsize=(figsize_inches, figsize_inches), dpi=dpi)

    # Use collection for faster rendering
    from matplotlib.collections import PatchCollection
    patches = []
    
    for x, y, a, b, angle in ellipses:
        ell = plt.matplotlib.patches.Ellipse((x, y), 2*a, 2*b,
                                             angle=np.degrees(angle),
                                             fill=True, edgecolor='white',  # Defect style: White
                                             facecolor='white')  # Defect style: White
        patches.append(ell)

    collection = PatchCollection(patches, match_original=True)
    ax.add_collection(collection)

    # Square domain limits
    half_size = DOMAIN_SIZE / 2.0
    ax.set_xlim(-half_size, half_size)
    ax.set_ylim(-half_size, half_size)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    ax.set_facecolor('black')  # Defect style: Black background

    filename = f'p{p:.1f}_seed_{seed}_Q11_{Q_values[0]:.4f}_Q12_{Q_values[1]:.4f}.png'
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # Save the initial image
    temp_path = os.path.join(output_subfolder, 'temp_' + filename)
    plt.savefig(temp_path, dpi=dpi, bbox_inches='tight', pad_inches=0, facecolor='black')
    plt.close(fig)
    
    # Convert to pure black and white
    img = Image.open(temp_path)
    img_array = np.array(img)
    
    # Use optimized conversion
    bw_array = convert_to_black_white_optimized(img_array)
    
    # Save final black and white image
    bw_img = Image.fromarray(bw_array)
    final_path = os.path.join(output_subfolder, filename)
    bw_img.save(final_path, optimize=True, quality=85)
    
    # Remove temporary file
    os.remove(temp_path)

def generate_images_optimized(num_images, major_axis, minor_axis, num_ellipses, p, p_idx, min_distance=0.01):
    """Optimized image generation with better resource management"""
    output_dir = 'output_ellipses'
    output_subfolder = os.path.join(output_dir, 'Square_Defect_Textures') # Save in its own folder
    os.makedirs(output_subfolder, exist_ok=True)

    Q_tensor_list = []

    for image_idx in range(1, num_images + 1):
        # Naming convention
        p_index = int(p * 10)
        seed = 1000000 + p_index * IMAGES_PER_P + image_idx # add + 1000000 here to have different seed names
        
        np.random.seed(seed)
        
        # Use optimized ellipse generation
        ellipses, angles = ellipse_params_optimized(
            major_axis, minor_axis, num_ellipses, p, 
            seed_offset=seed, min_distance=min_distance
        )
        
        # Use optimized Q-tensor calculation
        Q11, Q12, Q21, Q22 = calculate_q_tensor_optimized(angles)
        Q_tensor_list.append((seed, Q11, Q12, Q21, Q22))

        execution_time = time.time() - start_time
        
        # Use optimized plotting
        plot_ellipses_optimized(ellipses, seed, (Q11, Q12), 
                               execution_time, output_subfolder, N=250)
        
        # Progress reporting
        if image_idx % 10 == 0:
            print(f"  Generated {image_idx}/{num_images} images for p={p:.1f}")

    return Q_tensor_list

# Parameters - USING DEFECT SIZING ON SQUARE DOMAIN
PARAMS = {
    'major_axis': 0.11,
    'minor_axis': 0.01,
    'num_ellipses': 100,
    'num_images': 100,  # number of data per p-value
    'min_distance': 0.01  # Minimum distance between ellipses
}

# Main execution
if __name__ == '__main__':
    # Initialize CUDA for optimal performance
    setup_cuda()
    
    print(f"Starting optimized DEFECT generation on SQUARE domain")
    print(f"Domain size: {DOMAIN_SIZE} x {DOMAIN_SIZE}")
    print(f"Minimum distance: {PARAMS['min_distance']}")
    print(f"Using CUDA with {THREADS_PER_BLOCK} threads per block")
    
    total_start_time = time.time()
    
    for p_idx, p in enumerate(P_VALUES):
        p_start_time = time.time()
        print(f"Generating images (Defect Mode) labeled as p = {p:.1f}...")
        
        Q_tensors = generate_images_optimized(**PARAMS, p=p, p_idx=p_idx)
        
        p_time = time.time() - p_start_time
        print(f"Completed batch in {p_time:.2f} seconds")
    
    total_time = time.time() - total_start_time
    print(f"\nTotal execution time: {total_time:.2f} seconds")

### (b) Circular subdomain data:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import os
import time
from numba import cuda, jit
import math

# Start timing
start_time = time.time()

# Constants
RADIUS = 2.5 ## 2.5 : implicit boundary is this + 0.11 (ellispe semi major axis)


P_VALUES = np.round(np.array([0, 0.1, 0.4, 0.6, 1.0]), decimals=1)  # Limited p-values

IMAGES_PER_P = 1000  # match with num_images below !

# CUDA configuration - optimized for modern GPUs
THREADS_PER_BLOCK = 256

def setup_cuda():
    """Initialize CUDA for optimal performance"""
    cuda.select_device(0)
    # Optional: Enable fast math (trade precision for speed)
    # os.environ['NUMBA_CUDA_FAST_MATH'] = '1'

@cuda.jit(device=True)
def device_is_inside_ellipse_optimized(x, y, ellipse_params, min_distance):
    """Optimized ellipse collision detection with precomputed values"""
    x_center, y_center, a, b, angle = ellipse_params
    cos_angle = math.cos(angle)
    sin_angle = math.sin(angle)
    x_rot = (x - x_center) * cos_angle + (y - y_center) * sin_angle
    y_rot = -(x - x_center) * sin_angle + (y - y_center) * cos_angle
    
    # Use multiplication instead of division for better performance
    if min_distance > 0:
        effective_a = a + min_distance
        effective_b = b + min_distance
        # Compare squared values to avoid sqrt
        return (x_rot * x_rot) / (effective_a * effective_a) + (y_rot * y_rot) / (effective_b * effective_b) <= 1.0
    else:
        return (x_rot * x_rot) / (a * a) + (y_rot * y_rot) / (b * b) <= 1.0

@cuda.jit
def check_overlaps_kernel_optimized(ellipses, results, new_ellipse_idx, min_distance):
    """Optimized kernel with shared memory and precomputation"""
    i = cuda.grid(1)
    total_ellipses = ellipses.shape[0]
    
    if i < total_ellipses - 1:
        # Shared memory for the new ellipse - reduces global memory access
        new_ellipse_shared = cuda.shared.array(5, dtype=np.float32)
        tid = cuda.threadIdx.x
        
        # Load new ellipse into shared memory (first 5 threads do this)
        if tid < 5:
            new_ellipse_shared[tid] = ellipses[new_ellipse_idx, tid]
        cuda.syncthreads()
        
        # Get existing ellipse parameters
        existing_ellipse = (
            ellipses[i, 0], ellipses[i, 1], ellipses[i, 2],
            ellipses[i, 3], ellipses[i, 4]
        )

        overlap_found = False
        num_points = 75  # Reduced from original for speed, adjust as needed

        # Precompute rotation values once for the new ellipse
        new_cos_a = math.cos(new_ellipse_shared[4])
        new_sin_a = math.sin(new_ellipse_shared[4])
        
        # Precompute rotation values once for the existing ellipse
        existing_cos_a = math.cos(existing_ellipse[4])
        existing_sin_a = math.sin(existing_ellipse[4])

        # Check points on new ellipse against existing ellipses
        for j in range(num_points):
            theta = 2 * math.pi * j / num_points
            cos_t = math.cos(theta)
            sin_t = math.sin(theta)

            # Transform point on new ellipse
            x = (new_ellipse_shared[0] + 
                 new_ellipse_shared[2] * cos_t * new_cos_a - 
                 new_ellipse_shared[3] * sin_t * new_sin_a)
            y = (new_ellipse_shared[1] + 
                 new_ellipse_shared[2] * cos_t * new_sin_a + 
                 new_ellipse_shared[3] * sin_t * new_cos_a)

            if device_is_inside_ellipse_optimized(x, y, existing_ellipse, min_distance):
                overlap_found = True
                break

        # Check points on existing ellipse against new ellipse
        if not overlap_found:
            for j in range(num_points):
                theta = 2 * math.pi * j / num_points
                cos_t = math.cos(theta)
                sin_t = math.sin(theta)

                # Transform point on existing ellipse
                x = (existing_ellipse[0] + 
                     existing_ellipse[2] * cos_t * existing_cos_a - 
                     existing_ellipse[3] * sin_t * existing_sin_a)
                y = (existing_ellipse[1] + 
                     existing_ellipse[2] * cos_t * existing_sin_a + 
                     existing_ellipse[3] * sin_t * existing_cos_a)

                if device_is_inside_ellipse_optimized(x, y, 
                    (new_ellipse_shared[0], new_ellipse_shared[1], 
                     new_ellipse_shared[2], new_ellipse_shared[3], 
                     new_ellipse_shared[4]), min_distance):
                    overlap_found = True
                    break

        if overlap_found:
            results[i] = 1

@cuda.jit
def initialize_array_kernel(array, value):
    """Kernel to initialize device array with a value"""
    i = cuda.grid(1)
    if i < array.size:
        array[i] = value

def check_overlaps_cuda_optimized(ellipses, new_ellipse_idx, min_distance=0.0):
    """Optimized overlap checking with better memory management"""
    # Ensure contiguous array for better memory access
    ellipses_array = np.ascontiguousarray(ellipses, dtype=np.float32)
    d_ellipses = cuda.to_device(ellipses_array)
    d_results = cuda.device_array(len(ellipses)-1, dtype=np.int32)
    
    # Initialize results to 0 using a kernel
    blocks_per_grid_init = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    initialize_array_kernel[blocks_per_grid_init, THREADS_PER_BLOCK](d_results, 0)
    
    blocks_per_grid = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    
    # Use the optimized kernel
    check_overlaps_kernel_optimized[blocks_per_grid, THREADS_PER_BLOCK](
        d_ellipses, d_results, new_ellipse_idx, min_distance
    )
    
    results = d_results.copy_to_host()
    return np.any(results == 1)

@cuda.jit
def batch_check_overlaps_kernel(ellipses, results, new_ellipse_indices, min_distance):
    """Batch process multiple new ellipses in one kernel launch"""
    i = cuda.grid(1)
    total_ellipses = ellipses.shape[0]
    num_new_ellipses = new_ellipse_indices.shape[0]
    
    if i < total_ellipses:
        for new_idx in range(num_new_ellipses):
            new_ellipse_idx = new_ellipse_indices[new_idx]
            
            # Skip comparing with itself or invalid indices
            if i == new_ellipse_idx or new_ellipse_idx >= total_ellipses:
                continue
                
            existing_ellipse = (
                ellipses[i, 0], ellipses[i, 1], ellipses[i, 2],
                ellipses[i, 3], ellipses[i, 4]
            )
            new_ellipse = (
                ellipses[new_ellipse_idx, 0], ellipses[new_ellipse_idx, 1],
                ellipses[new_ellipse_idx, 2], ellipses[new_ellipse_idx, 3],
                ellipses[new_ellipse_idx, 4]
            )
            
            overlap_found = False
            num_points = 50  # Reduced for batch processing
            
            # Precompute rotations
            new_cos_a = math.cos(new_ellipse[4])
            new_sin_a = math.sin(new_ellipse[4])
            existing_cos_a = math.cos(existing_ellipse[4])
            existing_sin_a = math.sin(existing_ellipse[4])
            
            # Check overlap (simplified version for batch processing)
            for j in range(num_points):
                theta = 2 * math.pi * j / num_points
                cos_t = math.cos(theta)
                sin_t = math.sin(theta)
                
                # Check point on new ellipse
                x = new_ellipse[0] + new_ellipse[2] * cos_t * new_cos_a - new_ellipse[3] * sin_t * new_sin_a
                y = new_ellipse[1] + new_ellipse[2] * cos_t * new_sin_a + new_ellipse[3] * sin_t * new_cos_a
                
                if device_is_inside_ellipse_optimized(x, y, existing_ellipse, min_distance):
                    overlap_found = True
                    break
                    
                if not overlap_found:
                    # Check point on existing ellipse  
                    x = existing_ellipse[0] + existing_ellipse[2] * cos_t * existing_cos_a - existing_ellipse[3] * sin_t * existing_sin_a
                    y = existing_ellipse[1] + existing_ellipse[2] * cos_t * existing_sin_a + existing_ellipse[3] * sin_t * existing_cos_a
                    
                    if device_is_inside_ellipse_optimized(x, y, new_ellipse, min_distance):
                        overlap_found = True
                        break
            
            if overlap_found:
                # Use atomic add to avoid race conditions
                cuda.atomic.add(results, new_idx, 1)

def check_batch_overlaps_cuda(ellipses, new_ellipse_indices, min_distance=0.0):
    """Check overlaps for multiple new ellipses in batch"""
    if not new_ellipse_indices:
        return [False] * len(new_ellipse_indices)
    
    ellipses_array = np.ascontiguousarray(ellipses, dtype=np.float32)
    d_ellipses = cuda.to_device(ellipses_array)
    d_results = cuda.device_array(len(new_ellipse_indices), dtype=np.int32)
    d_indices = cuda.to_device(np.array(new_ellipse_indices, dtype=np.int32))
    
    # Initialize results to 0 using a kernel
    blocks_per_grid_init = (len(new_ellipse_indices) + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    initialize_array_kernel[blocks_per_grid_init, THREADS_PER_BLOCK](d_results, 0)
    
    blocks_per_grid = (len(ellipses) + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    
    batch_check_overlaps_kernel[blocks_per_grid, THREADS_PER_BLOCK](
        d_ellipses, d_results, d_indices, min_distance
    )
    
    results = d_results.copy_to_host()
    return [count > 0 for count in results]

@jit(nopython=True)
def fast_ellipse_point_generation(num_points, ellipse_params):
    """Precompute points on ellipse for faster collision detection"""
    x_center, y_center, a, b, angle = ellipse_params
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)
    
    points = np.empty((num_points, 2), dtype=np.float32)
    for j in range(num_points):
        theta = 2 * math.pi * j / num_points
        cos_t = math.cos(theta)
        sin_t = math.sin(theta)
        
        points[j, 0] = x_center + a * cos_t * cos_a - b * sin_t * sin_a
        points[j, 1] = y_center + a * cos_t * sin_a + b * sin_t * cos_a
    
    return points

def ellipse_params_optimized(major_axis, minor_axis, num_ellipses, p, domain_radius=RADIUS, seed_offset=0, min_distance=0.02):
    """Optimized ellipse generation with better GPU utilization"""
    ellipses = []
    angles = []
    max_attempts = 700

    full_range_upper = math.pi - 0.01
    biased_upper = (1 - p) * full_range_upper + p * (math.pi / 5)
    base_rotation = (seed_offset * 137.5) % math.pi
    sqrt_domain = math.sqrt(domain_radius)
    
    # Pre-allocate memory for temporary arrays
    temp_ellipses_list = []

    for attempt in range(max_attempts):
        if len(ellipses) >= num_ellipses:
            break

        #r = sqrt_domain * math.sqrt(np.random.random())
        r = domain_radius * math.sqrt(np.random.random()) 
        theta = np.random.random() * 2 * math.pi
        x_center, y_center = r * math.cos(theta), r * math.sin(theta)
        angle = (np.random.uniform(0, biased_upper) + base_rotation) % math.pi
        new_ellipse = (x_center, y_center, major_axis, minor_axis, angle)

        if len(ellipses) > 0:
            # Create temporary list for checking
            temp_ellipses = ellipses + [new_ellipse]
            temp_array = np.array(temp_ellipses, dtype=np.float32)
            
            # Use optimized overlap checking
            overlaps = check_overlaps_cuda_optimized(temp_array, len(temp_ellipses)-1, min_distance)
        else:
            overlaps = False

        if not overlaps:
            ellipses.append(new_ellipse)
            angles.append(angle)

    return ellipses, angles

@jit(nopython=True)
def calculate_q_tensor_optimized(angles):
    """Optimized Q-tensor calculation"""
    Q11 = 0.0
    Q12 = 0.0
    n = len(angles)

    for angle in angles:
        cos_theta = math.cos(2 * angle)  # Use double angle for direct calculation
        sin_theta = math.sin(2 * angle)
        Q11 += cos_theta
        Q12 += sin_theta

    Q11 /= (2.0 * n)  # Adjust for the double angle formula
    Q12 /= (2.0 * n)
    return Q11, Q12, Q12, -Q11

def convert_to_black_white_optimized(image_array, threshold=128):
    """
    Optimized black and white conversion
    """
    if len(image_array.shape) == 3:
        # Faster grayscale conversion using integer arithmetic
        image_gray = np.dot(image_array[...,:3], [299, 587, 114]) // 1000
    else:
        image_gray = image_array
    
    # Normalize efficiently
    if image_gray.max() > 1.0:
        min_val = image_gray.min()
        max_val = image_gray.max()
        if max_val > min_val:
            image_gray = ((image_gray - min_val) * 255 / (max_val - min_val)).astype(np.uint8)
    
    # Binary threshold
    binary_image = np.where(image_gray >= threshold, 255, 0).astype(np.uint8)
    
    return binary_image

def plot_ellipses_optimized(ellipses, seed, Q_values, execution_time, output_subfolder, N=250):  ## -------- Size of Images 250x250
    """Optimized ellipse plotting with INVERTED COLORS"""
    dpi = 100
    figsize_inches = N / dpi
    fig, ax = plt.subplots(figsize=(figsize_inches, figsize_inches), dpi=dpi)

    # Use collection for faster rendering
    from matplotlib.collections import PatchCollection
    patches = []
    
    for x, y, a, b, angle in ellipses:
        ell = plt.matplotlib.patches.Ellipse((x, y), 2*a, 2*b,
                                             angle=np.degrees(angle),
                                             fill=True, edgecolor='white',  # CHANGED: white ellipses
                                             facecolor='white')  # CHANGED: white ellipses
        patches.append(ell)

    collection = PatchCollection(patches, match_original=True)
    ax.add_collection(collection)
    
    ## +/- 0.37 used : (best since BetterG > EllipseMask - see 22ndDec2025_CompareBetterG_to_TrainingDataMask.ipynb)
    ## +/- 0.8 used here in addition to Radius par
    ax.set_xlim(-RADIUS-0.8, RADIUS+0.8)
    ax.set_ylim(-RADIUS-0.8, RADIUS+0.8) 
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    ax.set_facecolor('black')  # CHANGED: black background

    filename = f'p{p:.1f}_seed_{seed}_Q11_{Q_values[0]:.4f}_Q12_{Q_values[1]:.4f}.png'
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # Save the initial image
    temp_path = os.path.join(output_subfolder, 'temp_' + filename)
    plt.savefig(temp_path, dpi=dpi, bbox_inches='tight', pad_inches=0, facecolor='black')  # CHANGED: black background
    plt.close(fig)
    
    # Convert to pure black and white
    img = Image.open(temp_path)
    img_array = np.array(img)
    
    # Use optimized conversion
    bw_array = convert_to_black_white_optimized(img_array)
    
    # Save final black and white image
    bw_img = Image.fromarray(bw_array)
    final_path = os.path.join(output_subfolder, filename)
    bw_img.save(final_path, optimize=True, quality=85)  # Add compression for faster saving
    
    # Remove temporary file
    os.remove(temp_path)

def generate_images_optimized(num_images, major_axis, minor_axis, num_ellipses, p, p_idx, min_distance=0.01):
    """Optimized image generation with better resource management"""
    output_dir = 'output_ellipses'
    output_subfolder = os.path.join(output_dir, 'Train50k/Test500Data') # folder name where data is stored
    os.makedirs(output_subfolder, exist_ok=True)

    Q_tensor_list = []

    for image_idx in range(1, num_images + 1):
        # CHANGED: Use Code 2 seed naming convention
        p_index = int(p * 10)  # 0 for p=0.0, 1 for p=0.1, 2 for p=0.2, etc.
        seed = 1500000 + p_index * IMAGES_PER_P + image_idx  # ------------- add offset for test data here (replace 0 with another nmbr)
        
        np.random.seed(seed)
        
        # Use optimized ellipse generation
        ellipses, angles = ellipse_params_optimized(
            major_axis, minor_axis, num_ellipses, p, 
            seed_offset=seed, min_distance=min_distance
        )
        
        # Use optimized Q-tensor calculation
        Q11, Q12, Q21, Q22 = calculate_q_tensor_optimized(angles)
        Q_tensor_list.append((seed, Q11, Q12, Q21, Q22))

        execution_time = time.time() - start_time
        
        # Use optimized plotting
        plot_ellipses_optimized(ellipses, seed, (Q11, Q12), 
                               execution_time, output_subfolder, N=250) ######### ------------------------ Size of Images
        
        # Progress reporting
        if image_idx % 10 == 0:
            print(f"  Generated {image_idx}/{num_images} images for p={p:.1f}")

    return Q_tensor_list

# Parameters
## before running check that IMAGES_PER_P is THE SAME as "num_images" below (keeps naming convention consistent)

PARAMS = {
    'major_axis': 0.4,  
    'minor_axis': 0.04,      
    'num_ellipses': 100,
    'num_images': 1000,  # num_images*5 = total images
    'min_distance': 0.1  # Minimum distance between ellipses
}


# Main execution
if __name__ == '__main__':
    # Initialize CUDA for optimal performance
    setup_cuda()
    
    print(f"Starting optimized ellipse generation with minimum distance: {PARAMS['min_distance']}")
    print(f"Using CUDA with {THREADS_PER_BLOCK} threads per block")
    print(f"P-values: {P_VALUES}")  # Show which p-values are being used
    
    total_start_time = time.time()
    
    for p_idx, p in enumerate(P_VALUES):
        p_start_time = time.time()
        print(f"Generating images for p = {p:.1f}...")
        
        Q_tensors = generate_images_optimized(**PARAMS, p=p, p_idx=p_idx)
        
        p_time = time.time() - p_start_time
        print(f"Completed p = {p:.1f} in {p_time:.2f} seconds")
        print(f"  Generated {len(Q_tensors)} configurations")
    
    total_time = time.time() - total_start_time
    print(f"\nTotal execution time: {total_time:.2f} seconds")
    print(f"Average time per p-value: {total_time/len(P_VALUES):.2f} seconds")

### (b) Defect-type data on the circular subdomain:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import os
import time
from numba import cuda, jit
import math

# Start timing
start_time = time.time()

# Constants
RADIUS = 2.5 ## implicit boundary is this + 0.11 (ellispe semi major axis)

# P_VALUES will still be iterated over for file naming, 
# but they will no longer affect the orientation logic.
P_VALUES = np.round(np.array([0, 0.1, 0.4, 0.6, 1.0]), decimals=1) 

IMAGES_PER_P = 100  

# CUDA configuration - optimized for modern GPUs
THREADS_PER_BLOCK = 256

def setup_cuda():
    """Initialize CUDA for optimal performance"""
    try:
        cuda.select_device(0)
    except:
        print("CUDA device not found or already active")
    # Optional: Enable fast math (trade precision for speed)
    # os.environ['NUMBA_CUDA_FAST_MATH'] = '1'

@cuda.jit(device=True)
def device_is_inside_ellipse_optimized(x, y, ellipse_params, min_distance):
    """Optimized ellipse collision detection with precomputed values"""
    x_center, y_center, a, b, angle = ellipse_params
    cos_angle = math.cos(angle)
    sin_angle = math.sin(angle)
    x_rot = (x - x_center) * cos_angle + (y - y_center) * sin_angle
    y_rot = -(x - x_center) * sin_angle + (y - y_center) * cos_angle
    
    # Use multiplication instead of division for better performance
    if min_distance > 0:
        effective_a = a + min_distance
        effective_b = b + min_distance
        # Compare squared values to avoid sqrt
        return (x_rot * x_rot) / (effective_a * effective_a) + (y_rot * y_rot) / (effective_b * effective_b) <= 1.0
    else:
        return (x_rot * x_rot) / (a * a) + (y_rot * y_rot) / (b * b) <= 1.0

@cuda.jit
def check_overlaps_kernel_optimized(ellipses, results, new_ellipse_idx, min_distance):
    """Optimized kernel with shared memory and precomputation"""
    i = cuda.grid(1)
    total_ellipses = ellipses.shape[0]
    
    if i < total_ellipses - 1:
        # Shared memory for the new ellipse - reduces global memory access
        new_ellipse_shared = cuda.shared.array(5, dtype=np.float32)
        tid = cuda.threadIdx.x
        
        # Load new ellipse into shared memory (first 5 threads do this)
        if tid < 5:
            new_ellipse_shared[tid] = ellipses[new_ellipse_idx, tid]
        cuda.syncthreads()
        
        # Get existing ellipse parameters
        existing_ellipse = (
            ellipses[i, 0], ellipses[i, 1], ellipses[i, 2],
            ellipses[i, 3], ellipses[i, 4]
        )

        overlap_found = False
        num_points = 75  # Reduced from original for speed, adjust as needed

        # Precompute rotation values once for the new ellipse
        new_cos_a = math.cos(new_ellipse_shared[4])
        new_sin_a = math.sin(new_ellipse_shared[4])
        
        # Precompute rotation values once for the existing ellipse
        existing_cos_a = math.cos(existing_ellipse[4])
        existing_sin_a = math.sin(existing_ellipse[4])

        # Check points on new ellipse against existing ellipses
        for j in range(num_points):
            theta = 2 * math.pi * j / num_points
            cos_t = math.cos(theta)
            sin_t = math.sin(theta)

            # Transform point on new ellipse
            x = (new_ellipse_shared[0] + 
                 new_ellipse_shared[2] * cos_t * new_cos_a - 
                 new_ellipse_shared[3] * sin_t * new_sin_a)
            y = (new_ellipse_shared[1] + 
                 new_ellipse_shared[2] * cos_t * new_sin_a + 
                 new_ellipse_shared[3] * sin_t * new_cos_a)

            if device_is_inside_ellipse_optimized(x, y, existing_ellipse, min_distance):
                overlap_found = True
                break

        # Check points on existing ellipse against new ellipse
        if not overlap_found:
            for j in range(num_points):
                theta = 2 * math.pi * j / num_points
                cos_t = math.cos(theta)
                sin_t = math.sin(theta)

                # Transform point on existing ellipse
                x = (existing_ellipse[0] + 
                     existing_ellipse[2] * cos_t * existing_cos_a - 
                     existing_ellipse[3] * sin_t * existing_sin_a)
                y = (existing_ellipse[1] + 
                     existing_ellipse[2] * cos_t * existing_sin_a + 
                     existing_ellipse[3] * sin_t * existing_cos_a)

                if device_is_inside_ellipse_optimized(x, y, 
                    (new_ellipse_shared[0], new_ellipse_shared[1], 
                     new_ellipse_shared[2], new_ellipse_shared[3], 
                     new_ellipse_shared[4]), min_distance):
                    overlap_found = True
                    break

        if overlap_found:
            results[i] = 1

@cuda.jit
def initialize_array_kernel(array, value):
    """Kernel to initialize device array with a value"""
    i = cuda.grid(1)
    if i < array.size:
        array[i] = value

def check_overlaps_cuda_optimized(ellipses, new_ellipse_idx, min_distance=0.0):
    """Optimized overlap checking with better memory management"""
    # Ensure contiguous array for better memory access
    ellipses_array = np.ascontiguousarray(ellipses, dtype=np.float32)
    d_ellipses = cuda.to_device(ellipses_array)
    d_results = cuda.device_array(len(ellipses)-1, dtype=np.int32)
    
    # Initialize results to 0 using a kernel
    blocks_per_grid_init = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    initialize_array_kernel[blocks_per_grid_init, THREADS_PER_BLOCK](d_results, 0)
    
    blocks_per_grid = (len(ellipses)-1 + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    
    # Use the optimized kernel
    check_overlaps_kernel_optimized[blocks_per_grid, THREADS_PER_BLOCK](
        d_ellipses, d_results, new_ellipse_idx, min_distance
    )
    
    results = d_results.copy_to_host()
    return np.any(results == 1)

@cuda.jit
def batch_check_overlaps_kernel(ellipses, results, new_ellipse_indices, min_distance):
    """Batch process multiple new ellipses in one kernel launch"""
    i = cuda.grid(1)
    total_ellipses = ellipses.shape[0]
    num_new_ellipses = new_ellipse_indices.shape[0]
    
    if i < total_ellipses:
        for new_idx in range(num_new_ellipses):
            new_ellipse_idx = new_ellipse_indices[new_idx]
            
            # Skip comparing with itself or invalid indices
            if i == new_ellipse_idx or new_ellipse_idx >= total_ellipses:
                continue
                
            existing_ellipse = (
                ellipses[i, 0], ellipses[i, 1], ellipses[i, 2],
                ellipses[i, 3], ellipses[i, 4]
            )
            new_ellipse = (
                ellipses[new_ellipse_idx, 0], ellipses[new_ellipse_idx, 1],
                ellipses[new_ellipse_idx, 2], ellipses[new_ellipse_idx, 3],
                ellipses[new_ellipse_idx, 4]
            )
            
            overlap_found = False
            num_points = 50  # Reduced for batch processing
            
            # Precompute rotations
            new_cos_a = math.cos(new_ellipse[4])
            new_sin_a = math.sin(new_ellipse[4])
            existing_cos_a = math.cos(existing_ellipse[4])
            existing_sin_a = math.sin(existing_ellipse[4])
            
            # Check overlap (simplified version for batch processing)
            for j in range(num_points):
                theta = 2 * math.pi * j / num_points
                cos_t = math.cos(theta)
                sin_t = math.sin(theta)
                
                # Check point on new ellipse
                x = new_ellipse[0] + new_ellipse[2] * cos_t * new_cos_a - new_ellipse[3] * sin_t * new_sin_a
                y = new_ellipse[1] + new_ellipse[2] * cos_t * new_sin_a + new_ellipse[3] * sin_t * new_cos_a
                
                if device_is_inside_ellipse_optimized(x, y, existing_ellipse, min_distance):
                    overlap_found = True
                    break
                    
                if not overlap_found:
                    # Check point on existing ellipse  
                    x = existing_ellipse[0] + existing_ellipse[2] * cos_t * existing_cos_a - existing_ellipse[3] * sin_t * existing_sin_a
                    y = existing_ellipse[1] + existing_ellipse[2] * cos_t * existing_sin_a + existing_ellipse[3] * sin_t * existing_cos_a
                    
                    if device_is_inside_ellipse_optimized(x, y, new_ellipse, min_distance):
                        overlap_found = True
                        break
            
            if overlap_found:
                # Use atomic add to avoid race conditions
                cuda.atomic.add(results, new_idx, 1)

def check_batch_overlaps_cuda(ellipses, new_ellipse_indices, min_distance=0.0):
    """Check overlaps for multiple new ellipses in batch"""
    if not new_ellipse_indices:
        return [False] * len(new_ellipse_indices)
    
    ellipses_array = np.ascontiguousarray(ellipses, dtype=np.float32)
    d_ellipses = cuda.to_device(ellipses_array)
    d_results = cuda.device_array(len(new_ellipse_indices), dtype=np.int32)
    d_indices = cuda.to_device(np.array(new_ellipse_indices, dtype=np.int32))
    
    # Initialize results to 0 using a kernel
    blocks_per_grid_init = (len(new_ellipse_indices) + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    initialize_array_kernel[blocks_per_grid_init, THREADS_PER_BLOCK](d_results, 0)
    
    blocks_per_grid = (len(ellipses) + THREADS_PER_BLOCK - 1) // THREADS_PER_BLOCK
    
    batch_check_overlaps_kernel[blocks_per_grid, THREADS_PER_BLOCK](
        d_ellipses, d_results, d_indices, min_distance
    )
    
    results = d_results.copy_to_host()
    return [count > 0 for count in results]

@jit(nopython=True)
def fast_ellipse_point_generation(num_points, ellipse_params):
    """Precompute points on ellipse for faster collision detection"""
    x_center, y_center, a, b, angle = ellipse_params
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)
    
    points = np.empty((num_points, 2), dtype=np.float32)
    for j in range(num_points):
        theta = 2 * math.pi * j / num_points
        cos_t = math.cos(theta)
        sin_t = math.sin(theta)
        
        points[j, 0] = x_center + a * cos_t * cos_a - b * sin_t * sin_a
        points[j, 1] = y_center + a * cos_t * sin_a + b * sin_t * cos_a
    
    return points

def ellipse_params_optimized(major_axis, minor_axis, num_ellipses, p, domain_radius=RADIUS, seed_offset=0, min_distance=0.02):
    """
    MODIFIED: This now generates a 'hedgehog' defect independent of the 'p' parameter.
    """
    ellipses = []
    angles = []
    max_attempts = 700

    # sqrt_domain = math.sqrt(domain_radius) # Unused in original logic
    
    for attempt in range(max_attempts):
        if len(ellipses) >= num_ellipses:
            break

        # Generate random position within circle
        r = domain_radius * math.sqrt(np.random.random()) 
        theta_pos = np.random.random() * 2 * math.pi
        x_center, y_center = r * math.cos(theta_pos), r * math.sin(theta_pos)
        
        # -----------------------------------------------------------
        # CHANGED: HEDGEHOG DEFECT GENERATION
        # The angle is now strictly determined by the position (x, y).
        # atan2(y, x) aligns the major axis with the radius vector.
        # This ignores 'p' completely.
        # -----------------------------------------------------------
        angle = math.atan2(y_center, x_center)
        
        new_ellipse = (x_center, y_center, major_axis, minor_axis, angle)

        if len(ellipses) > 0:
            # Create temporary list for checking
            temp_ellipses = ellipses + [new_ellipse]
            temp_array = np.array(temp_ellipses, dtype=np.float32)
            
            # Use optimized overlap checking
            overlaps = check_overlaps_cuda_optimized(temp_array, len(temp_ellipses)-1, min_distance)
        else:
            overlaps = False

        if not overlaps:
            ellipses.append(new_ellipse)
            angles.append(angle)

    return ellipses, angles

@jit(nopython=True)
def calculate_q_tensor_optimized(angles):
    """Optimized Q-tensor calculation"""
    Q11 = 0.0
    Q12 = 0.0
    n = len(angles)

    for angle in angles:
        cos_theta = math.cos(2 * angle)  # Use double angle for direct calculation
        sin_theta = math.sin(2 * angle)
        Q11 += cos_theta
        Q12 += sin_theta

    Q11 /= (2.0 * n)  # Adjust for the double angle formula
    Q12 /= (2.0 * n)
    return Q11, Q12, Q12, -Q11

def convert_to_black_white_optimized(image_array, threshold=128):
    """
    Optimized black and white conversion
    """
    if len(image_array.shape) == 3:
        # Faster grayscale conversion using integer arithmetic
        image_gray = np.dot(image_array[...,:3], [299, 587, 114]) // 1000
    else:
        image_gray = image_array
    
    # Normalize efficiently
    if image_gray.max() > 1.0:
        min_val = image_gray.min()
        max_val = image_gray.max()
        if max_val > min_val:
            image_gray = ((image_gray - min_val) * 255 / (max_val - min_val)).astype(np.uint8)
    
    # Binary threshold
    binary_image = np.where(image_gray >= threshold, 255, 0).astype(np.uint8)
    
    return binary_image

def plot_ellipses_optimized(ellipses, seed, Q_values, execution_time, output_subfolder, N=250):  ## --------
    """Optimized ellipse plotting with INVERTED COLORS"""
    dpi = 100
    figsize_inches = N / dpi
    fig, ax = plt.subplots(figsize=(figsize_inches, figsize_inches), dpi=dpi)

    # Use collection for faster rendering
    from matplotlib.collections import PatchCollection
    patches = []
    
    for x, y, a, b, angle in ellipses:
        ell = plt.matplotlib.patches.Ellipse((x, y), 2*a, 2*b,
                                             angle=np.degrees(angle),
                                             fill=True, edgecolor='white',  # CHANGED: white ellipses
                                             facecolor='white')  # CHANGED: white ellipses
        patches.append(ell)

    collection = PatchCollection(patches, match_original=True)
    ax.add_collection(collection)
    
    ## +/- 0.37 used : (best since BetterG > EllipseMask - see 22ndDec2025_CompareBetterG_to_TrainingDataMask.ipynb)
    ax.set_xlim(-RADIUS-0.37, RADIUS+0.37)
    ax.set_ylim(-RADIUS-0.37, RADIUS+0.37) 
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')
    ax.set_facecolor('black')  # CHANGED: black background

    # Filename might contain misleading p value now since we generate hedgehogs regardless of p
    filename = f'p{p:.1f}_seed_{seed}_Q11_{Q_values[0]:.4f}_Q12_{Q_values[1]:.4f}.png'
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    # Save the initial image
    temp_path = os.path.join(output_subfolder, 'temp_' + filename)
    plt.savefig(temp_path, dpi=dpi, bbox_inches='tight', pad_inches=0, facecolor='black')  # CHANGED: black background
    plt.close(fig)
    
    # Convert to pure black and white
    img = Image.open(temp_path)
    img_array = np.array(img)
    
    # Use optimized conversion
    bw_array = convert_to_black_white_optimized(img_array)
    
    # Save final black and white image
    bw_img = Image.fromarray(bw_array)
    final_path = os.path.join(output_subfolder, filename)
    bw_img.save(final_path, optimize=True, quality=85)  # Add compression for faster saving
    
    # Remove temporary file
    os.remove(temp_path)

def generate_images_optimized(num_images, major_axis, minor_axis, num_ellipses, p, p_idx, min_distance=0.01):
    """Optimized image generation with better resource management"""
    output_dir = 'output_ellipses'
    output_subfolder = os.path.join(output_dir, 'Train/TestData_Defects')
    os.makedirs(output_subfolder, exist_ok=True)

    Q_tensor_list = []

    for image_idx in range(1, num_images + 1):
        # seed naming convention - 0 for p=0.0, 1 for p=0.1, 2 for p=0.2, etc.
        p_index = int(p * 10)  
        seed = 1000000 + p_index * IMAGES_PER_P + image_idx  # ------------- add offset for test data here (replace 0 with another nmbr)
        
        np.random.seed(seed)
        
        # Use optimized ellipse generation
        ellipses, angles = ellipse_params_optimized(
            major_axis, minor_axis, num_ellipses, p, 
            seed_offset=seed, min_distance=min_distance
        )
        
        # Use optimized Q-tensor calculation
        Q11, Q12, Q21, Q22 = calculate_q_tensor_optimized(angles)
        Q_tensor_list.append((seed, Q11, Q12, Q21, Q22))

        execution_time = time.time() - start_time
        
        # Use optimized plotting
        plot_ellipses_optimized(ellipses, seed, (Q11, Q12), 
                               execution_time, output_subfolder, N=250) ######### ------------------------ Size of Images
        
        # Progress reporting
        if image_idx % 10 == 0:
            print(f"  Generated {image_idx}/{num_images} images for p={p:.1f}")

    return Q_tensor_list

# Parameters
## before running check that IMAGES_PER_P is THE SAME as "num_images" below (keeps naming convention consistent)

PARAMS = {
    'major_axis': 0.4,  
    'minor_axis': 0.04,      
    'num_ellipses': 100,
    'num_images': 100,  # number of data per p-value (e.g., for 5 pvalues above, 5*num_images = total data)
    'min_distance': 0.1  # Minimum distance between ellipses
}


# Main execution
if __name__ == '__main__':
    # Initialize CUDA for optimal performance
    setup_cuda()
    
    print(f"Starting optimized ellipse generation with minimum distance: {PARAMS['min_distance']}")
    print(f"Using CUDA with {THREADS_PER_BLOCK} threads per block")
    print("NOTE: Generating HEDGEHOG DEFECTS only (independent of P_values)")
    
    total_start_time = time.time()
    
    for p_idx, p in enumerate(P_VALUES):
        p_start_time = time.time()
        print(f"Generating images (hedgehog mode) labeled as p = {p:.1f}...")
        
        Q_tensors = generate_images_optimized(**PARAMS, p=p, p_idx=p_idx)
        
        p_time = time.time() - p_start_time
        print(f"Completed batch in {p_time:.2f} seconds")
    
    total_time = time.time() - total_start_time
    print(f"\nTotal execution time: {total_time:.2f} seconds")